IMPORT LIBRARIES

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime

Feature 1 - The Transaction Parser


Load Dataset

In [25]:

df = pd.read_csv("DADS MP2 Dataset.csv")
df.info()

duplicate_count = 0
unparseable_amount = 0
unparseable_date =  0
total_transactions=df.shape[0]

def clean_dates():
  formats = [
      "%d/%m/%y",      # 12/04/24
      "%Y-%m-%d",      # 2024-04-12
      "%d-%b-%y",      # 12-Apr-24
      "%d %b %Y"       # 12 Apr 2024
  ]

  df["Date"] = df["Date"].astype(str)

  parsed_dates = pd.Series(pd.NaT, index=df.index)

  for fmt in formats:
      temp = pd.to_datetime(df["Date"], format=fmt, errors="coerce")
      parsed_dates = parsed_dates.fillna(temp)

  df["Date"] = parsed_dates



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         1328 non-null   object
 1   Time         1328 non-null   object
 2   Description  1328 non-null   object
 3   Type         1328 non-null   object
 4   Amount       1328 non-null   object
 5   Balance      1328 non-null   int64 
 6   Mode         1328 non-null   object
 7   Ref          1328 non-null   object
dtypes: int64(1), object(7)
memory usage: 83.1+ KB


In [26]:
def clean_amounts():

  df["Amount"] = df["Amount"].astype(str)

  df["Amount"] = (df["Amount"].str.replace("₹", "", regex=False).str.replace("Rs.", "", regex=False).str.replace(",", "", regex=False).str.strip())

  df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")   # amount


def standardize_type():
  df["Type"] = ( df["Type"].str.lower().replace({"dr": "debit","debit": "debit", "cr": "credit",      "credit": "credit"}))    # used df['Type'].unique() to know all the disntinct types present

def handling_duplicates_null():

  duplicate_count = df.duplicated().sum()
  df.drop_duplicates(inplace=True)
  df["Mode"] = df["Mode"].replace("", pd.NA)

  return duplicate_count



clean_dates()
clean_amounts()
standardize_type()
duplicate_count = handling_duplicates_null()

total_transactions=df.shape[0]

unparseable_amount = df["Amount"].isna().sum()
unparseable_date =  df["Date"].isna().sum()

print(f"Parsed {total_transactions} transactions across 6 months. Dropped {duplicate_count} duplicates. {unparseable_amount} unparseable amounts, {unparseable_date} unparseable dates.")
     

Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 743 unparseable dates.


Feature 2 - Vendor Extractor

In [28]:
vendor_mapping = {
    "Swiggy": ["SWIGGY", "BUNDL"],
    "Instamart": ["INSTAMART"],
    "Zomato": ["ZOMATO"],
    "Zepto": ["ZEPTO"],
    "Blinkit": ["BLINKIT"],
    "Amazon": ["AMAZON", "AMZN", "AMAZONPAY", "AMAZON IN", "AMAZONIN", "AMAZON SELLER"],
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA", "FSN E-COMMERCE"],
    "BigBasket": ["BIGBASKET"],
    "DMart": ["DMART", "AVENUE SUPERMARTS"],
    "Grofers": ["GROFERS"],
    "JioMart": ["KIRANAKART"],
    "Uber": ["UBER", "UBER SYSTEMS", "UBER INDIA", "UBER*TRIP"],
    "Ola": ["OLA", "ANI TECHNOLOGIES", "OLA ELECTRIC"],
    "Rapido": ["RAPIDO"],
    "BMTC": ["BMTC", "TUMMOC"],
    "Starbucks": ["STARBUCKS", "TATA STARBUCKS"],
    "Third Wave Coffee": ["THIRDWAVE", "THIRD WAVE", "TWC INDIA"],
    "Cafe Coffee Day": ["CCD", "CAFE COFFEE DAY", "COFFEE DAY"],
    "Truffles": ["TRUFFLES"],
    "Empire Restaurant": ["EMPIRE"],
    "Meghana Foods": ["MEGHANA"],
    "Restaurant": ["RESTAURANT", "DINEOUT", "BANGALORE RESTAURANT"],
    "BookMyShow": ["BOOKMYSHOW", "BIGTREE", "BMS"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Hotstar": ["HOTSTAR", "STAR INDIA", "DISNEY HOTSTAR"],
    "Jio": ["JIO", "JIOFIBER", "RELIANCE JIO"],
    "Airtel": ["AIRTEL", "BHARTI AIRTEL"],
    "Vi": ["VI", "VODAFONE IDEA"],
    "BESCOM": ["BESCOM", "BANGALORE ELEC SUPPLY"],
    "BWSSB": ["BWSSB", "WATER BILL"],
    "HP Petrol": ["HP PETROL"],
    "Indian Oil": ["INDIAN OIL", "IOC"],
    "BPCL": ["BPCL"],
    "Zerodha": ["ZERODHA", "COIN"],
    "Groww": ["GROWW", "GROWWPAY", "GROWW INVEST", "NEXTBILLION-GROWW"],
    "Salary": ["SALARY", "TECHCRUSH"],
    "Rent": ["RENT", "LANDLORD"],
    "Cash Withdrawal": ["ATM-WDL"],
    "P2P Transfer": ["AMAN", "ANKIT", "PRIYA", "NEHA", "SNEHA", "VIKAS", "KARAN"]
}


def extract_vendor(description):
    description = str(description).upper()

    for vendor, keywords in vendor_mapping.items():
        if any(keyword in description for keyword in keywords):
            return vendor

    return "uncategorised"

df["vendor_clean"] = df["Description"].apply(extract_vendor)


Feature 3 – Category Tagger

In [29]:
category_map = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Blinkit": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "Instamart": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",

    "Restaurant": "Restaurants",
    "Truffles": "Restaurants",
    "Empire Restaurant": "Restaurants",
    "Meghana Foods": "Restaurants",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Hotstar": "Subscriptions",

    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vi": "Utilities",

    "DMart": "Groceries",
    "BigBasket": "Groceries",
    "Grofers": "Groceries",
    "JioMart": "Groceries",

    "Groww": "Investments",
    "Zerodha": "Investments",

    "HP Petrol": "Fuel",
    "Indian Oil": "Fuel",
    "BPCL": "Fuel",

    "BookMyShow": "Entertainment",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",

    "Salary": "Income",
    "Rent": "Rent"
}

df["category"] = ( df["vendor_clean"].map(category_map).fillna("Uncategorised") )
df['category'].value_counts()


category
Food Delivery        344
Transport            236
E-commerce           172
Quick Commerce       118
Cafe                  99
Restaurants           73
Groceries             61
Utilities             47
Subscriptions         31
Fuel                  28
Investments           23
Uncategorised         22
Cash Withdrawal       17
Personal Transfer     14
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64

Feature 4 – Spending Overview

In [30]:
def spending_overview(df):

    transaction_summary = df.groupby("Type")["Amount"].sum()

    credits = transaction_summary.get("credit", 0)
    debits = transaction_summary.get("debit", 0)

    net_change = credits - debits
    if credits != 0:
      savings_rate = (( net_change / credits ) * 100)
    else:
      savings_rate = 0


    debit_df = df[df["Type"] == "debit"]

    top_categories = ( debit_df.groupby("category")["Amount"].sum().sort_values(ascending=False).head(5) )

    top_vendors = ( debit_df[debit_df["category"] != "Rent"].groupby("vendor_clean")["Amount"].sum().sort_values(ascending=False).head(5) )

    print("=" * 60)
    print("SPENDING OVERVIEW")
    print("=" * 60)

    print(f"Total Credits      : ₹{credits:,.2f}")
    print(f"Total Debits       : ₹{debits:,.2f}")
    print(f"Net Savings        : ₹{net_change:,.2f}")
    print(f"Savings Rate       : {savings_rate:.2f}%")
    print(f"Transactions       : {len(df)}")
    print(f"Unique Vendors     : {df['vendor_clean'].nunique()}")

    print("\nTop 5 Categories")
    print(top_categories)

    print("\nTop 5 Vendors")
    print(top_vendors)

spending_overview(df)


SPENDING OVERVIEW
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%
Transactions       : 1310
Unique Vendors     : 43

Top 5 Categories
category
E-commerce       603877.0
Investments      248160.0
Food Delivery    150839.0
Restaurants      117737.0
Rent             108000.0
Name: Amount, dtype: float64

Top 5 Vendors
vendor_clean
Amazon      328530.0
Zerodha     210000.0
Flipkart    177510.0
Swiggy       95523.0
Myntra       69529.0
Name: Amount, dtype: float64


Feature 5 – Monthly Trend Analysis

In [31]:
def monthly_category_trends(df):

    debit_df = df[df["Type"] == "debit"].copy()

    debit_df["Month"] = debit_df["Date"].dt.strftime("%b")   # Apr, May, Jun...

    month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    month_pivot = debit_df.pivot_table(
        values="Amount",
        index="category",
        columns="Month",
        aggfunc="sum",
        fill_value=0
    )

    # Arrange months chronologically
    existing_months = [m for m in month_order if m in month_pivot.columns]
    month_pivot = month_pivot[existing_months]

    print("=" * 70)
    print("CATEGORY × MONTH SPENDING MATRIX")
    print("=" * 70)
    print(month_pivot)

    first_month = month_pivot.iloc[:, 0]
    last_month = month_pivot.iloc[:, -1]

    growth = np.where( first_month == 0, np.nan, ((last_month - first_month) / first_month) * 100 )

    growth = pd.Series(growth, index=month_pivot.index).sort_values(ascending=False)

    print("\nBiggest Growth")
    print(f"{growth.idxmax()} : {growth.max():.2f}%")

    print("\nBiggest Decline")
    print(f"{growth.idxmin()} : {growth.min():.2f}%")

    return month_pivot, growth

month_matrix, growth = monthly_category_trends(df)


top5_categories = (
    month_matrix.sum(axis=1)
    .sort_values(ascending=False)
    .head(5)
    .index
)

     

CATEGORY × MONTH SPENDING MATRIX
Month                  Jan      Feb      Mar      Apr      May       Jun
category                                                                
Cafe                1528.0   2341.0   1706.0   2948.0   3815.0    1557.0
Cash Withdrawal        0.0   5000.0   8000.0   5000.0   5000.0    4000.0
E-commerce         43837.0  27870.0  71435.0  28010.0  38106.0  111543.0
Entertainment        311.0      0.0      0.0      0.0      0.0    1017.0
Food Delivery      11486.0   8518.0   8857.0  10204.0  10775.0   12471.0
Fuel               14157.0    627.0   9414.0   8350.0   5810.0       0.0
Groceries          13662.0   6378.0   1689.0   3328.0   5194.0    3559.0
Investments        19476.0      0.0  19883.0  19389.0      0.0    3834.0
Personal Transfer   4429.0   1860.0    266.0    663.0   2548.0    1295.0
Quick Commerce      4183.0   6931.0   8539.0   3800.0   2716.0    2503.0
Rent                   0.0  18000.0      0.0      0.0      0.0       0.0
Restaurants       

Feature 6 – Time of Day Analysis

In [32]:
def spending_time_analysis(df):

    debit_df = df[df["Type"] == "debit"].copy()

    # Extract hour from HH:MM
    debit_df["Hour"] = debit_df["Time"].str[:2].astype(int)

    # Category × Hour matrix  # Took Help of AI to print the trend in order of months tha is using reindex
    hour_matrix = ( debit_df.groupby(["category", "Hour"])["Amount"].sum().unstack(fill_value=0).reindex(columns=range(24), fill_value=0) )

    print("=" * 60)
    print("CATEGORY × HOUR SPENDING MATRIX")
    print("=" * 60)
    print(hour_matrix)

    # Peak hour for each category
    print("\nPeak Spending Hour")
    print("-" * 40)

    for category in hour_matrix.index:
        peak_hour = hour_matrix.loc[category].idxmax()
        print(f"{category:<20} {peak_hour:02d}:00")

spending_time_analysis(df)


CATEGORY × HOUR SPENDING MATRIX
Hour                    0        1        2        3        4        5   \
category                                                                  
Cafe                 872.0    549.0      0.0    316.0      0.0      0.0   
Cash Withdrawal        0.0      0.0      0.0      0.0      0.0      0.0   
E-commerce         18650.0   9801.0  10246.0  14865.0  12018.0  17258.0   
Entertainment        562.0    620.0      0.0      0.0      0.0      0.0   
Food Delivery        832.0   3017.0    928.0   1702.0   2618.0   3488.0   
Fuel                2675.0   2048.0      0.0    676.0   2010.0   2105.0   
Groceries           3895.0   1536.0   2352.0   7080.0   1169.0      0.0   
Investments         4883.0  15000.0      0.0      0.0  18834.0   4496.0   
Personal Transfer      0.0      0.0      0.0      0.0      0.0      0.0   
Quick Commerce      1542.0   1402.0   1143.0   1165.0    768.0   1448.0   
Rent                   0.0      0.0      0.0      0.0      0.0      

Feature 7 – Anomaly Detection

In [33]:
def detect_anomalies(df):
    """
    Detect unusually large transactions within each category
    using the z-score method.
    """

    debit_df = df[df["Type"] == "debit"].copy()

    # Mean and standard deviation for each category
    debit_df["mean"] = debit_df.groupby("category")["Amount"].transform("mean")
    debit_df["std"] = debit_df.groupby("category")["Amount"].transform("std")

    debit_df["std"] = debit_df["std"].replace(0, 1)

    # Z-score
    debit_df["z_score"] = ( (debit_df["Amount"] - debit_df["mean"]) / debit_df["std"] )

    # Transactions with z-score > 2
    anomalies = ( debit_df[debit_df["z_score"] > 2].sort_values("z_score", ascending=False)  )

    print("=" * 70)
    print("ANOMALOUS TRANSACTIONS")
    print("=" * 70)

    print( anomalies[ ["Date", "vendor_clean", "category", "Amount", "z_score"] ].head(5) )

    print(f"\nTotal Anomalies Detected: {len(anomalies)}")

    return anomalies

anomalies = detect_anomalies(df)


ANOMALOUS TRANSACTIONS
           Date vendor_clean     category   Amount   z_score
1298 2024-06-26       Amazon   E-commerce  22008.0  4.090349
269         NaT       Amazon   E-commerce  21986.0  4.085484
414         NaT   Restaurant  Restaurants   8383.0  3.884639
475  2024-03-05       Amazon   E-commerce  19917.0  3.627956
1271        NaT   Restaurant  Restaurants   7935.0  3.627582

Total Anomalies Detected: 33


Feature 8 – Spending Archetype Detection

In [34]:
def spending_archetypes(df):

    debit_df = df[df["Type"] == "debit"].copy()
    credit_df = df[df["Type"] == "credit"].copy()

    total_debits = debit_df["Amount"].sum()
    total_credits = credit_df["Amount"].sum()

    categories = debit_df.groupby("category")["Amount"].sum()

    archetypes = []


    # 1. The Foodie
    foodie = ( categories.get("Food Delivery", 0) + categories.get("Restaurants", 0) + categories.get("Cafe", 0) ) / total_debits * 100

    if foodie > 25:
        archetypes.append(f"THE FOODIE ({foodie:.1f}% of debits)")


    # 2. The Quick Commerce Junkie
    quick = categories.get("Quick Commerce", 0) / total_debits * 100

    if quick > 15:
        archetypes.append(f" THE QUICK COMMERCE JUNKIE ({quick:.1f}%)")


    # 3. The Shopaholic
    ecommerce = categories.get("E-commerce", 0) / total_debits * 100

    if ecommerce > 14:
        archetypes.append(f" THE SHOPAHOLIC ({ecommerce:.1f}%)")


    # 4. The Investor
    invest = categories.get("Investments", 0) / total_debits * 100

    if invest > 15:
        archetypes.append(f"THE INVESTOR ({invest:.1f}%)")


    # 5. The Late-Night Snacker
    food = debit_df[ debit_df["category"] == "Food Delivery"  ].copy()

    food["Hour"] = food["Time"].str[:2].astype(int)

    late = food[(food["Hour"] >= 21) | (food["Hour"] <= 2)]

    if len(food) > 0:

        late_percent = len(late) / len(food) * 100

        if late_percent > 50:
            archetypes.append(f" THE LATE-NIGHT SNACKER ({late_percent:.1f}% orders)")


    # 6. The Cab Commuter
    transport = categories.get("Transport", 0) / total_debits * 100

    if transport > 10:
        archetypes.append(f"THE CAB COMMUTER ({transport:.1f}%)")


    # 7. Subscription Lover
    subscriptions = debit_df[ debit_df["category"] == "Subscriptions" ]["vendor_clean"].nunique()

    if subscriptions >= 5:
        archetypes.append(f" THE SUBSCRIPTION LOVER ({subscriptions} subscriptions)")


    # 8. YOLO Spender
    savings_rate = (total_credits - total_debits) / total_credits * 100

    if savings_rate < 10:
        archetypes.append(f" THE YOLO SPENDER (Savings {savings_rate:.1f}%)")


    # 9. Disciplined Saver
    if savings_rate > 40:
        archetypes.append(f" THE DISCIPLINED SAVER (Savings {savings_rate:.1f}%)")

    # THE GENEROUS HELPER  (This is a new archtype invented by me)

    transfer_percent = ( categories.get("Personal Transfer", 0) / total_debits * 100  )

    transfer_count = len( debit_df[debit_df["category"] == "Personal Transfer"] )

    if transfer_percent > 2 and transfer_count >= 10:
        archetypes.append(f" THE GENEROUS HELPER ({transfer_percent:.1f}% of debits, {transfer_count} transfers)")
    print("=" * 60)
    print("SPENDING ARCHETYPES")
    print("=" * 60)

    if archetypes:
        for a in archetypes:
            print(a)
    else:
        print("No archetypes detected.")
    return archetypes

archetypes = spending_archetypes(df)

SPENDING ARCHETYPES
 THE SHOPAHOLIC (36.0%)
 THE YOLO SPENDER (Savings -229.3%)


Feature 9 – Final SpendDNA Report

In [35]:
def generate_report(df, anomalies, archetypes):
    debit_df = df[df["Type"] == "debit"]

    total_credit = df.loc[df["Type"] == "credit", "Amount"].sum()
    total_debit = debit_df["Amount"].sum()

    net = total_credit - total_debit
    savings_rate = (net / total_credit) * 100

    start_month = df["Date"].min().strftime("%b %Y") if pd.notna(df["Date"].min()) else "N/A"
    end_month = df["Date"].max().strftime("%b %Y") if pd.notna(df["Date"].max()) else "N/A"

    print("=" * 70)
    print(" " * 24 + "SpendDNA REPORT")
    print(" " * 10 + f"{start_month} - {end_month} | {len(df)} Transactions")
    print("=" * 70)

    # Executive Summary
    if savings_rate >= 0:
        savings_status = "🟢 POSITIVE SAVINGS"
    else:
        savings_status = "🔴 BURNING SAVINGS"

    print("\nEXECUTIVE SUMMARY")
    print("-" * 70)

    print(f"Total Credits      : ₹{total_credit:,.2f}")
    print(f"Total Debits       : ₹{total_debit:,.2f}")
    print(f"Net Savings        : ₹{net:,.2f}")
    print(f"Savings Rate       : {savings_rate:.2f}%    {savings_status} ")
    print(f"Transactions       : {len(df)}")
    print(f"Unique Vendors     : {df['vendor_clean'].nunique()}")

    # Top Categories
    
    print("\nTOP CATEGORIES")
    print("-" * 70)

    category = (
        debit_df.groupby("category")["Amount"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )

    for cat, amt in category.items():
        percent = amt / total_debit * 100
        bar = "#" * int(percent)
        print(f"{cat:<18} {bar:<20} {percent:5.1f}%   ₹{amt:,.0f}")

    # Top Vendors

    print("\nTOP VENDORS")
    print("-" * 70)

    vendors = (
        debit_df[debit_df["vendor_clean"] != "Rent"].groupby("vendor_clean")["Amount"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )

    for vendor, amt in vendors.items():
        count = (debit_df["vendor_clean"] == vendor).sum()
        print(f"{vendor:<18} ₹{amt:>10,.0f} ({count} transactions)")

    # Monthly Trend
    
    print("\nMONTHLY TREND")
    print("-" * 70)

    monthly = month_matrix.sum(axis=0)
    max_amt = monthly.max()

    for month, amt in monthly.items():
        bar = "#" * int((amt / max_amt) * 15)
        print(f"{month:<4} ₹{amt:>10,.0f} {bar}")

    # Top Anomalies

    print("\nTOP ANOMALIES")
    print("-" * 70)

    for _, row in anomalies.head(5).iterrows():
        date_str = row['Date'].strftime('%d %b') if pd.notna(row['Date']) else "Unknown"

        print(
            f"{date_str:<6} | "
            f"{row['vendor_clean']:<15} | "
            f"₹{row['Amount']:>8,.0f} | "
            f"z={row['z_score']:.2f}"
        )
    # Spending Archetypes


    print("\nRAHUL'S SPENDING ARCHETYPES")
    print("-" * 70)

    for archetype in archetypes:
        print(f"✓ {archetype}")

    # Key Insights

    print("\nKEY INSIGHTS")
    print("-" * 70)

    food = ( category.get("Food Delivery", 0) + category.get("Restaurants", 0) + category.get("Cafe", 0) ) / total_debit * 100
    ecommerce = ( debit_df.groupby("category")["Amount"].sum().get("E-commerce", 0) ) / total_debit * 100

    print(f"• Savings rate is {savings_rate:.1f}%.")
    print(f"• Food + Restaurants + Cafe account for {food:.1f}% of total spending.")
    print(f"• E-commerce contributes {ecommerce:.1f}% of total spending.")
    print(f"• Highest spending vendor: {vendors.index[0]} (₹{vendors.iloc[0]:,.0f}).")
    print(f"• {len(anomalies)} anomalous transactions were detected.")
    print("=" * 70)

generate_report(df, anomalies, archetypes)

                        SpendDNA REPORT
          Jan 2024 - Jun 2024 | 1310 Transactions

EXECUTIVE SUMMARY
----------------------------------------------------------------------
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%    🔴 BURNING SAVINGS 
Transactions       : 1310
Unique Vendors     : 43

TOP CATEGORIES
----------------------------------------------------------------------
E-commerce         ###################################  36.0%   ₹603,877
Investments        ##############        14.8%   ₹248,160
Food Delivery      ########               9.0%   ₹150,839
Restaurants        #######                7.0%   ₹117,737
Rent               ######                 6.4%   ₹108,000

TOP VENDORS
----------------------------------------------------------------------
Amazon             ₹   328,530 (86 transactions)
Zerodha            ₹   210,000 (14 transactions)
Flipkart           ₹   177,510 (47 trans